In [1]:
from datasets import load_dataset
from openai import OpenAI

import pandas as pd
import os, re
import time
import google.generativeai as genai
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


# Load the dataset

In [2]:
dataset = load_dataset("tum-nlp/cognitive-biases-in-llms", split='train')

df = pd.DataFrame(dataset)
df_conf = df[df['bias'] == 'Confirmation Bias'].sample(n=5)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Setup prompt

In [3]:
samples = []
for _, row in df_conf.iterrows():
  samples.append({
    "scenario": row["scenario"],
    "prompt_control": (
      f"Scenario: {row['scenario']}\n\n"
      f"{row['control']}\n"
      "Respond with exactly one integer between 0 and 10, inclusive "
      "(valid values: 0,1,2,3,4,5,6,7,8,9,10). No additional text."
    ),
    "prompt_treatment": (
      f"Scenario: {row['scenario']}\n\n"
      f"{row['treatment']}\n"
      "Respond with exactly one integer between 0 and 10, inclusive "
      "(valid values: 0,1,2,3,4,5,6,7,8,9,10). No additional text."
    ),
  })

# Setup clients

In [ ]:
# Client for GPT, DeepSeek and Grok
azure_client = OpenAI(
  base_url="https://models.inference.ai.azure.com",
  api_key=os.getenv("OPENAI_API_KEY")
)

# Client for Gemini
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel('gemini-2.5-flash')

MODELS = {
  "GPT": ("azure", "GPT-5-mini"),
  "DeepSeek": ("azure", "DeepSeek-R1"),
  "Grok": ("azure", "Grok-3"),
  "Gemini": ("gemini", "gemini-2.5-flash"),
}

# Querying the models

In [5]:
def query_llm(model_key, prompt):
  backend, model_name = MODELS[model_key]
  try:
    if backend == "azure": # GPT, DeepSeek and Grok
      response = azure_client.chat.completions.create(
        model=model_name,
        messages=[
          {"role": "system", "content": "Answer only with an integer."},
          {"role": "user", "content": prompt}
        ],
        stream=False
      )
      text = response.choices[0].message.content.strip()
    else:  # Gemini
      response = gemini_model.generate_content(prompt)
      text = response.text.strip()

    # Extract the number from the answer
    match = re.search(r'\d+', text)
    if match:
      return int(match.group())
    print(f"Warning ({model_key}): Non-numeric answer '{text}'")
    return None
  except Exception as e:
    print(f"Error ({model_key}): {e}")
    return None

# Response collection

In [6]:
results = []

for i, sample in enumerate(samples):
  print(f"Sample {i+1}")

  for model_key in MODELS:
    print(f"Model {model_key}")

    control_val = query_llm(model_key, sample["prompt_control"])
    time.sleep(5)
    treatment_val = query_llm(model_key, sample["prompt_treatment"])
    time.sleep(5)

    results.append({
      "sample": i + 1,
      "model": model_key,
      "control": control_val,
      "treatment": treatment_val,
    })
    print(f"control={control_val}, treatment={treatment_val}")


df_results = pd.DataFrame(results)
df_results

Sample 1
Model GPT
control=10, treatment=10
Model DeepSeek
Error (DeepSeek): Error code: 429 - {'error': {'code': 'RateLimitReached', 'message': 'Rate limit of 1 per 60s exceeded for UserByModelByMinute. Please wait 59 seconds before retrying.', 'details': 'Rate limit of 1 per 60s exceeded for UserByModelByMinute. Please wait 59 seconds before retrying.'}}
control=10, treatment=None
Model Grok
control=10, treatment=10
Model Gemini
control=10, treatment=10
Sample 2
Model GPT
control=10, treatment=10
Model DeepSeek
Error (DeepSeek): Error code: 429 - {'error': {'code': 'RateLimitReached', 'message': 'Rate limit of 1 per 60s exceeded for UserByModelByMinute. Please wait 0 seconds before retrying.', 'details': 'Rate limit of 1 per 60s exceeded for UserByModelByMinute. Please wait 0 seconds before retrying.'}}
Error (DeepSeek): Error code: 429 - {'error': {'code': 'RateLimitReached', 'message': 'Rate limit of 1 per 60s exceeded for UserByModelByMinute. Please wait 0 seconds before retrying.

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 935.64ms


control=10, treatment=10


,sample,model,control,treatment
0,1,GPT,10.0,10.0
1,1,DeepSeek,10.0,NaN
2,1,Grok,10.0,10.0
3,1,Gemini,10.0,10.0
4,2,GPT,10.0,10.0
5,2,DeepSeek,NaN,NaN
6,2,Grok,10.0,10.0
7,2,Gemini,10.0,10.0
8,3,GPT,NaN,NaN
9,3,DeepSeek,NaN,NaN


# Calculating the confirmation bias

In [7]:
# Metrics per sample (Dataset 1)
import pandas as pd

# Bias = (control - treatment) / max(control, treatment)
def calc_bias(control, treatment):
  if pd.isna(control) or pd.isna(treatment):
    return None
  if max(control, treatment) == 0:
    return 0.0
  return (control - treatment) / max(control, treatment)

# Confirmatory rule for Dataset 1: treatment < control
def classify_confirmation(control, treatment):
  if pd.isna(control) or pd.isna(treatment):
    return "missing"
  if treatment < control:
    return "confirmatory"
  if treatment > control:
    return "disconfirmatory"
  return "neutral"

df_results["bias"] = df_results.apply(
  lambda r: calc_bias(r["control"], r["treatment"]), axis=1
)

df_results["confirmation_type"] = df_results.apply(
  lambda r: classify_confirmation(r["control"], r["treatment"]), axis=1
)

# Binary confirmatory variable used for CBR in your definition
df_results["confirmatory"] = (df_results["confirmation_type"] == "confirmatory").astype(int)

# Aggregate metrics for each model
summary_rows = []
for model, g in df_results.groupby("model"):
  g_valid = g.dropna(subset=["control", "treatment"])
  n_total = len(g_valid)
  n_confirm = int((g_valid["confirmation_type"] == "confirmatory").sum())
  n_disconfirm = int((g_valid["confirmation_type"] == "disconfirmatory").sum())

  # CBR = Nconfirm / Ntotal
  cbr = (n_confirm / n_total) if n_total > 0 else None

  # CBI = (Nconfirm - Ndisconfirm) / (Nconfirm + Ndisconfirm)
  denom_cbi = n_confirm + n_disconfirm
  cbi = ((n_confirm - n_disconfirm) / denom_cbi) if denom_cbi > 0 else None

  summary_rows.append({
    "model": model,
    "avg_bias": g_valid["bias"].mean() if n_total > 0 else None,
    "CBR": cbr,
    "CBI": cbi
  })

metrics_per_model = pd.DataFrame(summary_rows).sort_values("model").reset_index(drop=True)

print("Average metrics for each model (Bias, CBR, CBI)")
print(metrics_per_model.to_string(index=False))

Average metrics for each model (Bias, CBR, CBI)
   model  avg_bias  CBR  CBI
DeepSeek       NaN  NaN None
     GPT       0.0  0.0 None
  Gemini       0.0  0.0 None
    Grok       0.0  0.0 None
